# 2 — The same SAE, on a real model

Notebook 1 graded the SAE against features we planted. Here there is no answer key:
these are GPT-2's own activations, and nobody knows what its true features are.

What changes is only where the data comes from. The `SAE` class is byte-for-byte the
one you already read.

Two things to look for:

1. **Do the latents mean anything?** You judge by reading the text that makes each one
   fire — the only tool available once ground truth is gone.
2. **How good is yours, really?** At the end you load a production SAE trained on far
   more data and compare. The gap is the honest part of this notebook.

> **Cost warning.** The training cell below runs the GPU flat out for about
> **2–3 minutes** and your laptop will get hot. Every other cell is cheap. If you'd
> rather not train at all, set `SKIP_TRAINING = True` and the notebook runs entirely
> on the pretrained SAE.

In [ ]:
import os, warnings, torch, numpy as np
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERLENS_ALLOW_MPS"] = "1"   # verified numerically identical to CPU
from IPython.display import HTML, display

from activations import load_activations
from sae import SAE, train_sae, sae_metrics
from featureviz import top_activating_examples, render_examples

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

acts, meta = load_activations("acts_layer6.pt")
sequences  = torch.load("acts_layer6_seqs.pt")
print(f"{acts.shape[0]:,} activations of width {acts.shape[1]} from {meta['hook_name']}")
print(f"stored as {acts.dtype} = {acts.numel()*acts.element_size()/1e9:.2f} GB, kept on CPU")
print(f"training device: {DEVICE}")

If `acts_layer6.pt` is missing, build it once with:

```bash
python build_cache.py --n 500000
```

That runs GPT-2 over ~3,900 passages of text and keeps the layer-6 residual stream at
every token position. It takes about two minutes and writes 733 MB. Everything below
reads from that file, so you can retrain SAEs all afternoon without touching GPT-2
again.

## Train your SAE

768 dimensions in, 4096 latents out — a 5.3× overcomplete dictionary.

**This uses TopK rather than an L1 penalty**, and the reason is practical. On real
activations L1 needs calibrating, and the useful range is nowhere near notebook 1's.
Measured on this exact cache at 400 steps:

| λ | resulting L0 |
|---|---|
| 0.5 | 1112 |
| 1.0 | 875 |
| 2.0 | 593 |
| 4.0 | 342 |

All far too dense — you want tens of active latents, not hundreds, and finding the
right λ means a search you'd pay for in GPU minutes each time. `k=32` just declares
the answer. Notebook 1 showed the catch (TopK needs you to know the true sparsity);
for LM residual streams ~32 is the conventional choice, and it's what production SAEs
target.

*(~2–3 min on the GPU, fans up. It saves itself, so re-running the cell is free.)*

In [ ]:
from pathlib import Path

SKIP_TRAINING = False  # 🔧 True = use only the pretrained SAE, no GPU load at all

K         = 32     # 🔧 latents allowed to fire at once — this IS the sparsity
STEPS     = 1_500  # 🔧 the main quality knob; 5_000 is better and ~8 min
N_LATENTS = 4096   # 🔧 dictionary size; 8192 doubles both quality and time

sae, ckpt = None, Path(f"sae_layer6_k{K}_n{N_LATENTS}_s{STEPS}.pt")

if not SKIP_TRAINING:
    sae = SAE(d_in=acts.shape[1], n_latents=N_LATENTS, k=K, seed=0).to(DEVICE)
    if ckpt.exists():
        sae.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print(f"loaded {ckpt} — no GPU time spent")
    else:
        import time; t0 = time.time()
        train_sae(sae, acts, l1_coeff=0.0, steps=STEPS, batch_size=2048, lr=3e-4, seed=0)
        torch.save(sae.state_dict(), ckpt)
        print(f"trained in {(time.time()-t0)/60:.1f} min and saved {ckpt}")
else:
    print("SKIP_TRAINING is on — jump to the pretrained SAE section below")

In [ ]:
mine = sae_metrics(sae, acts[:100_000].to(DEVICE).float())
print(f"L0                   {mine.l0:.1f} of {N_LATENTS} latents active per token")
print(f"                     (exactly K by construction — that is what TopK buys you)")
print(f"variance unexplained {mine.fraction_variance_unexplained:.1%}")
print(f"dead latents         {mine.n_dead:,} ({mine.n_dead/N_LATENTS:.0%} of the dictionary)")

## Read the features

A latent is only a column index until you look at what fires it. Below: the token that
triggered each latent most strongly, highlighted, with its context.

This is the actual practice of SAE interpretability — there is no metric here, you read
the examples and decide whether a story holds.

In [ ]:
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained("gpt2", device="cpu")
decode = lambda t: model.to_string(torch.tensor([t]))

with torch.no_grad():
    z = sae.encode(acts[:200_000].to(DEVICE).float()).cpu()

fire_rate = (z > 0).float().mean(0)
alive = (fire_rate > 1e-4).nonzero().flatten()
print(f"{len(alive):,} of {N_LATENTS} latents fire on at least 1 in 10,000 tokens")

In [ ]:
# 🔧 re-run for a different random sample of latents
for latent in alive[torch.randperm(len(alive))[:8]].tolist():
    examples = top_activating_examples(z, sequences, latent=latent, top_n=4, context=7)
    display(HTML(f"<div style='color:#666;font-size:11px;font-family:sans-serif'>"
                 f"fires on {fire_rate[latent]:.2%} of tokens</div>"
                 + render_examples(examples, decode_token=decode, latent=latent)))

### Hunting for a specific feature

The other direction: pick text containing a concept, see which latent it drives hardest,
then check what else that latent responds to. Agreement across both directions is what
makes a feature claim credible.

In [ ]:
PROBE = " Paris"   # 🔧 try " Bridge", " 1984", " def", " Monday", " Dr"

probe_tokens = model.to_tokens(f"The capital of France is{PROBE}", prepend_bos=True)
_, cache = model.run_with_cache(probe_tokens, names_filter=meta["hook_name"])
with torch.no_grad():
    probe_z = sae.encode(cache[meta["hook_name"]][0, -1].to(DEVICE).float()).cpu()

for latent in probe_z.topk(3).indices.tolist():
    display(HTML(f"<div style='color:#666;font-size:11px;font-family:sans-serif'>"
                 f"activation on '{PROBE}' = {probe_z[latent]:.2f}</div>"
                 + render_examples(
                       top_activating_examples(z, sequences, latent=latent, top_n=4, context=7),
                       decode_token=decode, latent=latent)))

## Yours vs. a production SAE

Joseph Bloom's `gpt2-small-res-jb` was trained on the same layer of the same model, with
orders of magnitude more compute. Same browser, same code — only the SAE differs.

Read the two side by side. The metrics tell you part of it; the examples tell you the
rest.

In [ ]:
from sae_lens import SAE as LensSAE
ref = LensSAE.from_pretrained("gpt2-small-res-jb", meta["hook_name"], device=DEVICE)

with torch.no_grad():
    sample = acts[:100_000].to(DEVICE).float()
    ref_z = ref.encode(sample)
    ref_recon = ref.decode(ref_z)
    ref_fvu = ((ref_recon - sample)**2).sum().item() / ((sample - sample.mean(0))**2).sum().item()
    ref_l0 = (ref_z > 0).float().sum(-1).mean().item()

print(f"{'':22}{'yours':>12}{'Bloom':>12}")
print(f"{'latents':22}{N_LATENTS:>12,}{ref.cfg.d_sae:>12,}")
print(f"{'L0 (active/token)':22}{mine.l0:>12.1f}{ref_l0:>12.1f}")
print(f"{'variance unexplained':22}{mine.fraction_variance_unexplained:>11.1%}{ref_fvu:>12.1%}")
print(f"{'dead latents':22}{mine.n_dead:>12,}{'—':>12}")

In [ ]:
with torch.no_grad():
    ref_z_full = ref.encode(acts[:200_000].to(DEVICE).float()).cpu()

ref_rate = (ref_z_full > 0).float().mean(0)
ref_alive = (ref_rate > 1e-4).nonzero().flatten()
for latent in ref_alive[torch.randperm(len(ref_alive))[:6]].tolist():
    display(HTML("<div style='color:#666;font-size:11px;font-family:sans-serif'>"
                 f"Bloom SAE · fires on {ref_rate[latent]:.2%} of tokens</div>"
                 + render_examples(
                       top_activating_examples(ref_z_full, sequences, latent=latent, top_n=4, context=7),
                       decode_token=decode, latent=latent)))

## Steering: features are causal, not just descriptive

Everything so far has been observational — this latent *correlates* with that text. The
stronger claim is that the direction is what the model actually uses.

Test it by force: add a latent's decoder direction into the residual stream mid-forward
and see whether the generated text bends toward it. If it does, you've found something
the model computes with, not just something that co-occurs.

In [ ]:
LATENT   = int(alive[torch.randperm(len(alive))[0]])   # 🔧 pick one you recognised above
STRENGTH = 20.0                                        # 🔧 crank it up
PROMPT   = "The best thing about the weekend is"

display(HTML(render_examples(
    top_activating_examples(z, sequences, latent=LATENT, top_n=3, context=7),
    decode_token=decode, latent=LATENT)))

direction = sae.W_dec[LATENT].detach().cpu()

def steer(activation, hook):
    activation[:, :, :] = activation + STRENGTH * direction
    return activation

torch.manual_seed(0)
plain = model.generate(PROMPT, max_new_tokens=25, temperature=0.7, verbose=False)
with model.hooks(fwd_hooks=[(meta["hook_name"], steer)]):
    torch.manual_seed(0)
    steered = model.generate(PROMPT, max_new_tokens=25, temperature=0.7, verbose=False)

print("unsteered:", plain)
print("\nsteered  :", steered)

## 🔧 Where to go next

1. **`STEPS = 5_000`.** The single biggest lever on feature quality — about 8 minutes,
   and the dead-latent count should drop noticeably. Notebook 1 showed under-training
   silently merges features; the same happens here, you just have no way to measure it.
   (Watch your machine's temperature; this is the cell that heats it.)
2. **Change `K`** to 8 or 128 and re-read the features. Low k forces each latent to carry
   more meaning; high k lets the SAE spread a concept thinly across many latents.
3. **Change the layer.** Rebuild with `python build_cache.py --layer 2 --out acts_layer2.pt`.
   Early layers lean lexical, later layers more semantic — see it for yourself.
4. **Hunt for a polysemantic neuron.** Take a raw GPT-2 dimension, find its top-activating
   text, and confirm it mixes unrelated things — then find the SAE latents that split it.
5. **Steer harder.** Push `STRENGTH` to 50 or 100. Where does it stop producing English,
   and what does that say about how much the model relies on that direction?

Your `saeplay.py` in this folder does the next thing along: using SAE latents as features
for a downstream probe. Note it was written against an older `sae_lens` — `SAE.from_pretrained`
now returns the SAE alone, not a `(sae, cfg, sparsity)` tuple.